<a href="https://colab.research.google.com/github/ncinsli/CLIP-classification-experiments/blob/main/main.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import transformers
import torch
import requests
import torchvision
from torchvision import transforms
from PIL import Image
from transformers import CLIPModel, CLIPProcessor
from tqdm.notebook import tqdm
import matplotlib.pyplot as plt
from sklearn import metrics, preprocessing
import torch.nn.functional as F
import gc

In [ ]:
model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32", dtype=torch.bfloat16, attn_implementation="sdpa").to('cuda')
processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")

In [ ]:
BATCH_SIZE = 128

In [ ]:
url = "http://images.cocodataset.org/val2017/000000039769.jpg"
image = Image.open(requests.get(url, stream=True).raw)
labels = ["a photo of a cat", "a photo of a dog", "a photo of a car"]

inputs = processor(text=labels, images=image, return_tensors="pt", padding=True).to('cuda')

outputs = model(**inputs)
logits_per_image = outputs.logits_per_image
probs = logits_per_image.softmax(dim=1)


In [ ]:
print(f'Probits: {probs[0].tolist()}')

### Imagenette dataset

In [ ]:
def collate_fn(batch):
  images = [item[0] for item in batch]
  labels = [item[1] for item in batch]

  return images, labels

classes = ['tench', 'English springer', 'cassette player', 'chain saw', 'church', 'French horn', 'garbage truck', 'gas pump', 'golf ball', 'parachute']

transform = transforms.Compose([
    # transforms.Resize((600, 800)),
    transforms.PILToTensor(),
])

imagenette_data = torchvision.datasets.Imagenette('imagenette/', download=True, transform=transform)
data_loader = torch.utils.data.DataLoader(imagenette_data, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, collate_fn=collate_fn)

In [ ]:
plt.set_cmap('gray')
plt.imshow(data_loader.dataset[torch.randint(len(imagenette_data), (1,))][0][0])

In [ ]:
image = data_loader.dataset[128][0]
plt.imshow(image[0])

In [ ]:
inputs = processor(text=['A man with a fish', 'Respectable man', 'Epstein island', 'Well-grown fish'], images=image, padding=True, return_tensors='pt').to('cuda')
outputs = model(**inputs)

In [ ]:
outputs.logits_per_image.softmax(dim=1).squeeze().tolist()

In [ ]:
inputs['input_ids']

Considering this output, we see that CLIP gets a sequence of sentences as a count x vector_dim Tensor.

This is yet simple encoding mechanism, below we have much better embeddings

In [ ]:
raise BaseException("Use this exception for making cell below work")

BaseException: Use this exception for making cell below work

In [ ]:
torch.cuda.empty_cache()
gc.collect()

186

In [ ]:
model.get_text_features(inputs['input_ids']).pooler_output.detach()

In [ ]:
correct_classifications = 0
all_classifications = 0
predictions = torch.Tensor([], device='cpu')
truth = torch.Tensor([], device='cpu')

for batch, t in tqdm(data_loader):
  with torch.no_grad():
    inputs = processor(images=batch, text=classes, padding=True, return_tensors='pt').to('cuda')
    outputs = model(**inputs)
    predicted_cat = outputs.logits_per_image.argmax(dim=1).to('cpu')

    batch_truth = torch.Tensor(t).to('cpu')
    predictions = torch.cat((predictions, predicted_cat))
    truth = torch.cat((truth, batch_truth))

    correct_classifications += int((predicted_cat == batch_truth).sum())
    all_classifications += BATCH_SIZE # Useful when we stop the cycle


  0%|          | 0/74 [00:00<?, ?it/s]

In [ ]:
print(correct_classifications, all_classifications, correct_classifications / all_classifications)

9354 9472 0.9875422297297297


In [ ]:
print(f'Accuracy: {metrics.accuracy_score(truth, predictions)}')
print(f'Precision: {metrics.precision_score(truth, predictions, average='macro')}')

Accuracy: 0.9878551061358116
Precision: 0.9877990798430517
